In [2]:
#from pathlib import Path

#from intrarepresentational_alignment.data import load_metanet

In [3]:
#DATA_PATH = Path("data/mr-en-20250924142102.rdf")

In [4]:
#repo = load_metanet(DATA_PATH)
#print(f"Loaded  {len(repo.metaphors):,} metaphors, {len(repo.frames):,} frames, "
#      f"{len(repo.mappings):,} mappings, {len(repo.examples):,} examples")

## LCC Metaphor Corpus — Source & Target Domain Kernel Matrices

In [5]:
from collections import Counter
from pathlib import Path

from intrarepresentational_alignment.data import load_lcc

LCC_PATH = Path("data/en_small.xml")
instances = load_lcc(LCC_PATH)
print(f"Loaded {len(instances):,} LCC instances")

# Collect (source_expr, target_expr) co-occurrence pairs from each instance.
# Each instance's <LmSource> and <LmTarget> tags define one concrete mapping.
pair_counts = Counter(
    (s, t)
    for inst in instances
    for s in inst.source_expressions
    for t in inst.target_expressions
)

TOP_N = 60
pairs = pair_counts.most_common(TOP_N)
source_terms = [s for (s, t), _ in pairs]
target_terms = [t for (s, t), _ in pairs]

print(f"\nTop-{TOP_N} (source → target) pairs by frequency:")
for (s, t), count in pairs[:10]:
    print(f"  {s!r:20s} → {t!r:20s}  (n={count})")

Loaded 16,265 LCC instances

Top-60 (source → target) pairs by frequency:
  'burden'             → 'tax'                 (n=24)
  'is'                 → 'money'               (n=23)
  'have'               → 'money'               (n=20)
  'is'                 → 'government'          (n=19)
  'are'                → 'government'          (n=18)
  'chronic'            → 'poverty'             (n=16)
  'live in'            → 'poverty'             (n=15)
  'is'                 → 'tax'                 (n=15)
  'not'                → 'money'               (n=15)
  'increase'           → 'understanding'       (n=15)


In [6]:
import numpy as np

from intrarepresentational_alignment.embedding import Embedder
from intrarepresentational_alignment.models import EmbeddingModel
from intrarepresentational_alignment.alignment import cka

embedder = Embedder(EmbeddingModel.ALL_MINILM_L6_V2)

S_embs = embedder.embed(source_terms)  # (N, D)
T_embs = embedder.embed(target_terms)  # (N, D)

# Intra-domain kernel matrices (each is NxN)
K_S = S_embs @ S_embs.T  # source kernel
K_T = T_embs @ T_embs.T  # target kernel

score = cka(K_S, K_T)
print(f"CKA(K_source, K_target) = {score:.4f}")

c:\Users\afjal\ICL Code\Intra\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\afjal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 61.51it/s]
BertModel LOAD REPORT from: 

CKA(K_source, K_target) = 0.2153


In [7]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, K, terms, title in [
    (axes[0], K_S, source_terms, "K_source  (source expressions)"),
    (axes[1], K_T, target_terms, "K_target  (target expressions)"),
]:
    im = ax.imshow(K, cmap="viridis", vmin=K.min(), vmax=K.max(), aspect="auto")
    ax.set_xticks(range(len(terms)))
    ax.set_yticks(range(len(terms)))
    ax.set_xticklabels(terms, rotation=90, fontsize=6)
    ax.set_yticklabels(terms, fontsize=6)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f"CKA(K_source, K_target) = {score:.4f}  |  top-{TOP_N} S→T pairs",
    fontsize=12,
)
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import matplotlib.pyplot as plt

from intrarepresentational_alignment.alignment import cka_permutation_test

result = cka_permutation_test(K_S, K_T, n_permutations=1000, rng=np.random.default_rng(42))

print(f"Observed CKA : {result.observed:.4f}")
print(f"Null mean    : {result.null.mean():.4f}  ± {result.null.std():.4f}")
print(f"p-value      : {result.p_value:.4f}  ({'significant' if result.p_value < 0.05 else 'not significant'} at α=0.05)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(result.null, bins=40, color="steelblue", alpha=0.8, label="Null distribution")
ax.axvline(result.observed, color="red", linewidth=2, label=f"Observed CKA = {result.observed:.4f}")
ax.set_xlabel("CKA")
ax.set_ylabel("Count")
ax.set_title(f"CKA permutation test  (n={len(result.null)} permutations)  p = {result.p_value:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

## Per-domain CKA experiments

In [ ]:
from collections import Counter
from dataclasses import dataclass

import numpy as np

from intrarepresentational_alignment.alignment import cka_permutation_test, PermutationTestResult
from intrarepresentational_alignment.embedding import Embedder
from intrarepresentational_alignment.models import EmbeddingModel

MIN_PAIRS = 5       # skip domains too sparse to give a meaningful kernel
N_PERMS   = 500     # permutations per domain

embedder = Embedder(EmbeddingModel.ALL_MINILM_L6_V2)

domain_results: dict[str, PermutationTestResult] = {}
domain_n: dict[str, int] = {}

all_concepts = sorted({inst.target_concept for inst in instances})

for concept in all_concepts:
    pair_counts = Counter(
        (s, t)
        for inst in instances
        if inst.target_concept == concept
        for s in inst.source_expressions
        for t in inst.target_expressions
    )
    if len(pair_counts) < MIN_PAIRS:
        print(f"  SKIP {concept} ({len(pair_counts)} pairs < {MIN_PAIRS})")
        continue

    src = [s for s, _ in pair_counts]
    tgt = [t for _, t in pair_counts]

    S = embedder.embed(src)
    T = embedder.embed(tgt)

    result = cka_permutation_test(
        S @ S.T, T @ T.T,
        n_permutations=N_PERMS,
        rng=np.random.default_rng(42),
    )
    domain_results[concept] = result
    domain_n[concept] = len(pair_counts)
    sig = "*" if result.p_value < 0.05 else ""
    print(f"{concept:30s}  n={len(pair_counts):4d}  CKA={result.observed:.3f}  p={result.p_value:.3f} {sig}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Sort by observed CKA descending
sorted_domains = sorted(domain_results, key=lambda d: domain_results[d].observed, reverse=True)
cka_scores = [domain_results[d].observed for d in sorted_domains]
p_values   = [domain_results[d].p_value  for d in sorted_domains]
ns         = [domain_n[d]                for d in sorted_domains]

colors = ["#e74c3c" if p < 0.05 else "#95a5a6" for p in p_values]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(sorted_domains)), cka_scores, color=colors, edgecolor="white")

# Annotate with n and p-value
for i, (score, p, n) in enumerate(zip(cka_scores, p_values, ns)):
    ax.text(score + 0.002, i, f"n={n}  p={p:.3f}", va="center", fontsize=7)

ax.set_yticks(range(len(sorted_domains)))
ax.set_yticklabels(sorted_domains, fontsize=8)
ax.set_xlabel("CKA(K_source, K_target)", fontsize=10)
ax.set_title("Source–Target kernel alignment per metaphorical domain\n"
             "(red = significant at α=0.05, permutation test)", fontsize=11)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlim(left=0)

sig_patch   = mpatches.Patch(color="#e74c3c", label="p < 0.05")
insig_patch = mpatches.Patch(color="#95a5a6", label="p ≥ 0.05")
ax.legend(handles=[sig_patch, insig_patch], loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()